In [ ]:
import os
import glob
import json
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

os.makedirs("/workspace/models/ncf", exist_ok=True)
os.makedirs("/workspace/outputs/metrics", exist_ok=True)
os.makedirs("/workspace/outputs/figures", exist_ok=True)

print("TensorFlow:", tf.__version__)
print("CPU:", tf.config.list_physical_devices("CPU"))
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
def read_single_csv_from_spark_folder(folder):
    files = glob.glob(f"{folder}/part-*.csv")
    if not files:
        raise FileNotFoundError(f"No part csv found in {folder}")
    return pd.read_csv(files[0])

train_df = read_single_csv_from_spark_folder("/workspace/outputs/ncf/train_csv")
test_df = read_single_csv_from_spark_folder("/workspace/outputs/ncf/test_csv")

with open("/workspace/outputs/ncf/ncf_metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

num_users = int(metadata["num_users"])
num_movies = int(metadata["num_movies"])
genome_feature_dim = int(metadata["genome_feature_dim"])
movie_genome_features = np.load("/workspace/outputs/ncf/movie_genome_features.npy").astype("float32")

train_df.head(), test_df.head(), metadata, movie_genome_features.shape

In [ ]:
BATCH_SIZE = 8192

train_ds = tf.data.Dataset.from_tensor_slices((
    {
        "user_idx": train_df["user_idx"].astype("int32").values,
        "movie_idx": train_df["movie_idx"].astype("int32").values,
    },
    train_df["label"].astype("float32").values
)).shuffle(200_000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((
    {
        "user_idx": test_df["user_idx"].astype("int32").values,
        "movie_idx": test_df["movie_idx"].astype("int32").values,
    },
    test_df["label"].astype("float32").values
)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
from tensorflow.keras import layers, Model

EMBEDDING_DIM = 64
GENOME_PROJECTION_DIM = 64

user_input = layers.Input(shape=(), name="user_idx", dtype=tf.int32)
movie_input = layers.Input(shape=(), name="movie_idx", dtype=tf.int32)

user_embedding = layers.Embedding(
    input_dim=num_users,
    output_dim=EMBEDDING_DIM,
    name="user_embedding"
)(user_input)

movie_embedding = layers.Embedding(
    input_dim=num_movies,
    output_dim=EMBEDDING_DIM,
    name="movie_embedding"
)(movie_input)

genome_embedding = layers.Embedding(
    input_dim=num_movies,
    output_dim=genome_feature_dim,
    weights=[movie_genome_features],
    trainable=False,
    name="movie_genome_features"
)(movie_input)

user_vec = layers.Flatten(name="user_vec")(user_embedding)
movie_vec = layers.Flatten(name="movie_vec")(movie_embedding)
genome_vec = layers.Flatten(name="genome_vec")(genome_embedding)

genome_projection = layers.Dense(GENOME_PROJECTION_DIM, activation="relu", name="genome_projection")(genome_vec)
genome_projection = layers.Dropout(0.2)(genome_projection)

x = layers.Concatenate(name="hybrid_user_movie_genome_concat")([user_vec, movie_vec, genome_projection])
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.2)(x)
x = layers.Dense(64, activation="relu")(x)
x = layers.Dropout(0.2)(x)
x = layers.Dense(32, activation="relu")(x)

output = layers.Dense(1, activation="sigmoid", name="preference_probability")(x)

model = Model(inputs=[user_input, movie_input], outputs=output)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=2,
        restore_best_weights=True
    )
]

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=10,
    callbacks=callbacks
)

In [ ]:
eval_result = model.evaluate(test_ds, return_dict=True)
eval_result

In [ ]:
model.save("/workspace/models/ncf/ncf_model.keras")

ncf_metrics = {
    "model": "Hybrid Neural Collaborative Filtering with Movie Genome Features",
    "embedding_dim": int(EMBEDDING_DIM),
    "genome_projection_dim": int(GENOME_PROJECTION_DIM),
    "genome_feature_dim": int(genome_feature_dim),
    "uses_genome_features": True,
    "batch_size": int(BATCH_SIZE),
    **{k: float(v) for k, v in eval_result.items()}
}

with open("/workspace/outputs/metrics/ncf_metrics.json", "w", encoding="utf-8") as f:
    json.dump(ncf_metrics, f, ensure_ascii=False, indent=2)

ncf_metrics

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="train_loss")
plt.plot(history.history["val_loss"], label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title("Quá trình huấn luyện Hybrid NCF với movie genome features")
plt.legend()
plt.tight_layout()
plt.savefig("/workspace/outputs/figures/ncf_training_loss.png", dpi=200)
plt.show()